# RVC Model Activation Comparison

Tento notebook umožňuje:
1. Načíst dva RVC modely
2. Extrahovat aktivace při průchodu audia
3. Najít největší rozdíly v aktivacích
4. Zkopírovat váhy vrstev s největšími rozdíly

In [ ]:
# Buňka 1: Importy a konfigurace
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
from typing import Dict, List, Tuple

# Přidání cesty k RVC modulům
sys.path.insert(0, os.getcwd())

from rvc.lib.algorithm.synthesizers import Synthesizer

# Konfigurace
MODEL_PATH_1 = "Ravdess_F_Angry_60e_1140s.pth"
MODEL_PATH_2 = "Anton.pth"
TOP_SCALARS = 1000  # Počet největších skalárních rozdílů k analýze
TOP_N_LAYERS = 5    # Počet vrstev ke kopírování (s nejvíce rozdíly)
DEVICE = "mps"

print(f"Device: {DEVICE}")
print(f"Model 1: {MODEL_PATH_1}")
print(f"Model 2: {MODEL_PATH_2}")
print(f"Top skalárů k analýze: {TOP_SCALARS}")
print(f"Top vrstev ke kopírování: {TOP_N_LAYERS}")

In [42]:
# Buňka 2: Funkce pro načtení modelu

def load_rvc_model(model_path: str, device: str = "cpu") -> Tuple[Synthesizer, dict]:
    """
    Načte RVC model z .pth souboru.
    
    Args:
        model_path: Cesta k .pth souboru
        device: Zařízení (cpu/cuda)
    
    Returns:
        Tuple[Synthesizer, dict]: Model a checkpoint dict
    """
    cpt = torch.load(model_path, map_location="cpu")
    
    # Extrakce konfigurace
    config = cpt["config"]
    use_f0 = cpt.get("f0", 1)
    version = cpt.get("version", "v1")
    text_enc_hidden_dim = 768 if version == "v2" else 256
    
    # Aktualizace speaker dim
    config[-3] = cpt["weight"]["emb_g.weight"].shape[0]
    
    # Vytvoření modelu
    model = Synthesizer(
        *config,
        use_f0=use_f0,
        text_enc_hidden_dim=text_enc_hidden_dim,
        is_half=False
    )
    
    # Odstranění enc_q (není potřeba pro inferenci)
    del model.enc_q
    
    # Načtení vah
    model.load_state_dict(cpt["weight"], strict=False)
    model.eval().to(device)
    
    print(f"Načten model: {model_path}")
    print(f"  - Sample rate: {cpt['sr']}")
    print(f"  - Version: {version}")
    print(f"  - Use F0: {use_f0}")
    
    return model, cpt

# Načtení obou modelů
model1, cpt1 = load_rvc_model(MODEL_PATH_1, DEVICE)
model2, cpt2 = load_rvc_model(MODEL_PATH_2, DEVICE)

Načten model: Ravdess_F_Angry_60e_1140s.pth
  - Sample rate: 48000
  - Version: v2
  - Use F0: True
Načten model: Ravdess_F_Neutral_111e_1110s.pth
  - Sample rate: 48000
  - Version: v2
  - Use F0: True


In [43]:
# Buňka 3: Hook systém pro zachycení aktivací

class ActivationCapture:
    """
    Třída pro zachycení aktivací ze všech vrstev modelu.
    """
    
    def __init__(self, model: torch.nn.Module):
        self.activations: Dict[str, torch.Tensor] = {}
        self.hooks = []
        self._register_hooks(model)
    
    def _register_hooks(self, model: torch.nn.Module):
        """Registruje forward hooks na všechny vrstvy."""
        for name, module in model.named_modules():
            if name == "":  # Skip root module
                continue
            hook = module.register_forward_hook(self._make_hook(name))
            self.hooks.append(hook)
    
    def _make_hook(self, name: str):
        """Vytvoří hook funkci pro danou vrstvu."""
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                self.activations[name] = output.detach().cpu()
            elif isinstance(output, tuple) and len(output) > 0:
                # Některé vrstvy vrací tuple
                if isinstance(output[0], torch.Tensor):
                    self.activations[name] = output[0].detach().cpu()
        return hook
    
    def clear(self):
        """Vyčistí zachycené aktivace."""
        self.activations.clear()
    
    def remove_hooks(self):
        """Odstraní všechny hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

print("ActivationCapture třída definována.")

ActivationCapture třída definována.


In [44]:
# Buňka 4: Načtení audia a příprava vstupů pro inferenci

import librosa
import torch.nn.functional as F
from rvc.lib.utils import load_embedding

# ===== KONFIGURACE AUDIA =====
AUDIO_PATH = "Angry++.wav"  # <-- ZMĚŇTE NA VAŠE AUDIO
EMBEDDER_MODEL = "contentvec"  # nebo "hubert"

# ===== Načtení Hubert/ContentVec modelu =====
print("Načítám embedder model...")
hubert_model = load_embedding(EMBEDDER_MODEL, None)
hubert_model.to(DEVICE)
hubert_model.eval()

# ===== Funkce pro přípravu vstupů z audia =====
def prepare_inputs_from_audio(
    audio_path: str,
    hubert_model,
    target_sr: int = 16000,
    device: str = "cpu"
) -> dict:
    """
    Připraví vstupy pro RVC model z audio souboru.
    
    Args:
        audio_path: Cesta k audio souboru
        hubert_model: Načtený Hubert/ContentVec model
        target_sr: Cílový sample rate pro Hubert (16000)
        device: Zařízení
    
    Returns:
        dict s phone, phone_lengths, pitch, nsff0, sid
    """
    # Načtení audia
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    print(f"Načteno audio: {audio_path}")
    print(f"  - Délka: {len(audio)/sr:.2f}s, Sample rate: {sr}")
    
    # Převod na tensor
    audio_tensor = torch.from_numpy(audio).float().unsqueeze(0).to(device)
    
    # Extrakce features pomocí Hubert
    with torch.no_grad():
        feats = hubert_model(audio_tensor)["last_hidden_state"]
    
    # Feature upsampling (2x) - stejně jako v pipeline.py
    feats = F.interpolate(feats.permute(0, 2, 1), scale_factor=2).permute(0, 2, 1)
    
    seq_len = feats.shape[1]
    print(f"  - Feature sequence length: {seq_len}")
    
    # Generování pitch (zjednodušená verze - pro přesné použijte rmvpe/crepe)
    pitch = torch.zeros(1, seq_len, dtype=torch.long, device=device)
    nsff0 = torch.zeros(1, seq_len, dtype=torch.float, device=device)
    
    return {
        "phone": feats,
        "phone_lengths": torch.tensor([seq_len], device=device),
        "pitch": pitch,
        "nsff0": nsff0,
        "sid": torch.tensor([0], device=device)
    }

# ===== Příprava vstupů =====
inputs = prepare_inputs_from_audio(AUDIO_PATH, hubert_model, device=DEVICE)

# ===== Zachycení aktivací z obou modelů =====
print("\nZachycuji aktivace z modelu 1...")
capture1 = ActivationCapture(model1)
with torch.no_grad():
    output1 = model1.infer(**inputs)
activations1 = dict(capture1.activations)
capture1.remove_hooks()

print("Zachycuji aktivace z modelu 2...")
capture2 = ActivationCapture(model2)
with torch.no_grad():
    output2 = model2.infer(**inputs)
activations2 = dict(capture2.activations)
capture2.remove_hooks()

print(f"\nZachyceno {len(activations1)} aktivací z modelu 1")
print(f"Zachyceno {len(activations2)} aktivací z modelu 2")

Načítám embedder model...
Načteno audio: Angry++.wav
  - Délka: 3.10s, Sample rate: 16000
  - Feature sequence length: 308

Zachycuji aktivace z modelu 1...
Zachycuji aktivace z modelu 2...

Zachyceno 433 aktivací z modelu 1
Zachyceno 433 aktivací z modelu 2


In [ ]:
# Buňka 5: Výpočet rozdílů mezi aktivacemi (OPTIMALIZOVANĚ na úrovni skalárů)

def compute_scalar_differences(
    activations1: Dict[str, torch.Tensor],
    activations2: Dict[str, torch.Tensor],
    top_k: int = 1000
) -> Tuple[torch.Tensor, torch.Tensor, List[str], List[int]]:
    """
    Najde top K největších skalárních rozdílů napříč všemi vrstvami.
    Plně vektorizovaná implementace.
    
    Returns:
        - top_values: tensor top K hodnot
        - top_indices: tensor globálních indexů
        - layer_names: seznam názvů vrstev pro každý index
        - layer_offsets: offset pro každou vrstvu v globálním tensoru
    """
    common_layers = sorted(set(activations1.keys()) & set(activations2.keys()))
    print(f"Počet společných vrstev: {len(common_layers)}")
    
    # 1. Spočítej rozdíly a spoj do jednoho tensoru
    all_diffs = []
    layer_info = []  # (layer_name, start_idx, end_idx)
    current_idx = 0
    
    for layer_name in common_layers:
        act1 = activations1[layer_name].float()
        act2 = activations2[layer_name].float()
        
        if act1.shape != act2.shape:
            continue
        
        diff = torch.abs(act1 - act2).flatten()
        all_diffs.append(diff)
        
        layer_info.append((layer_name, current_idx, current_idx + diff.numel()))
        current_idx += diff.numel()
    
    # 2. Jeden velký tensor
    all_diffs_tensor = torch.cat(all_diffs)
    print(f"Celkem skalárů: {all_diffs_tensor.numel():,}")
    
    # 3. Jeden torch.topk() - rychlé!
    k = min(top_k, all_diffs_tensor.numel())
    top_values, top_indices = torch.topk(all_diffs_tensor, k)
    
    # 4. Mapování indexů zpět na vrstvy
    top_layers = []
    for idx in top_indices.tolist():
        for layer_name, start, end in layer_info:
            if start <= idx < end:
                top_layers.append(layer_name)
                break
    
    return top_values, top_indices, top_layers, layer_info

# Výpočet rozdílů
top_values, top_indices, top_layers, layer_info = compute_scalar_differences(
    activations1, activations2, TOP_SCALARS
)

# Počet výskytů každé vrstvy v top rozdílech
from collections import Counter
layer_counts = Counter(top_layers)

print(f"\nTop {TOP_SCALARS} skalárních rozdílů pochází z {len(layer_counts)} vrstev:")
print("-" * 70)
print(f"{'#':<4} {'Vrstva':<45} {'Počet':<8} {'%':<8}")
print("-" * 70)
for i, (layer, count) in enumerate(layer_counts.most_common(20)):
    pct = count / TOP_SCALARS * 100
    print(f"{i+1:<4} {layer:<45} {count:<8} {pct:.1f}%")

if len(layer_counts) > 20:
    print(f"... a dalších {len(layer_counts) - 20} vrstev")

In [ ]:
# Buňka 6: Identifikace vah pro TOP N vrstev

def layer_to_weight_keys(layer_name: str, state_dict: dict) -> List[str]:
    """
    Najde klíče vah ve state_dict pro danou vrstvu.
    """
    matching_keys = []
    for key in state_dict.keys():
        if key.startswith(layer_name + ".") or key == layer_name + ".weight" or key == layer_name + ".bias":
            matching_keys.append(key)
        elif key == layer_name:
            matching_keys.append(key)
    return matching_keys

# Získání TOP N vrstev (seřazených podle počtu výskytů)
top_n_layers = layer_counts.most_common(TOP_N_LAYERS)
unique_layers = [layer for layer, _ in top_n_layers]

print(f"Top {TOP_N_LAYERS} vrstev ke kopírování:")
print("=" * 70)

total_weights = 0
for layer, count in top_n_layers:
    weight_keys = layer_to_weight_keys(layer, cpt1["weight"])
    total_weights += len(weight_keys)
    pct = count / TOP_SCALARS * 100
    print(f"\n{layer}")
    print(f"  Skalárů v top {TOP_SCALARS}: {count} ({pct:.1f}%)")
    print(f"  Weight tensory:")
    for wk in weight_keys:
        shape = cpt1["weight"][wk].shape
        print(f"    → {wk}: {list(shape)}")

print(f"\n" + "=" * 70)
print(f"Celkem: {TOP_N_LAYERS} vrstev, {total_weights} weight tensorů")

# Info o vynechaných vrstvách
skipped = len(layer_counts) - TOP_N_LAYERS
if skipped > 0:
    skipped_count = sum(c for _, c in layer_counts.most_common()[TOP_N_LAYERS:])
    print(f"Vynecháno: {skipped} vrstev ({skipped_count} skalárů = {skipped_count/TOP_SCALARS*100:.1f}%)")

In [ ]:
# Buňka 7: Vizualizace

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Graf 1: Top vrstvy podle počtu skalárů v top rozdílech
ax1 = axes[0]
top_n = min(15, len(layer_counts))
layers = [l[:30] + "..." if len(l) > 30 else l for l, _ in layer_counts.most_common(top_n)]
counts = [c for _, c in layer_counts.most_common(top_n)]
ax1.barh(range(len(layers)), counts, color='steelblue')
ax1.set_yticks(range(len(layers)))
ax1.set_yticklabels(layers, fontsize=9)
ax1.set_xlabel(f'Počet skalárů v top {TOP_SCALARS}')
ax1.set_title(f'Vrstvy s největšími rozdíly aktivací')
ax1.invert_yaxis()

# Graf 2: Distribuce hodnot top rozdílů
ax2 = axes[1]
ax2.hist(top_values.numpy(), bins=50, color='coral', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Absolutní rozdíl')
ax2.set_ylabel('Počet')
ax2.set_title(f'Distribuce top {TOP_SCALARS} rozdílů')
ax2.axvline(x=top_values.median().item(), color='red', linestyle='--', 
            label=f'Medián: {top_values.median().item():.4f}')
ax2.legend()

plt.tight_layout()
plt.show()

# Pie chart - rozložení vrstev
fig, ax = plt.subplots(figsize=(10, 8))
top_for_pie = 10
labels = [l[:25] + "..." if len(l) > 25 else l for l, _ in layer_counts.most_common(top_for_pie)]
sizes = [c for _, c in layer_counts.most_common(top_for_pie)]
other = TOP_SCALARS - sum(sizes)
if other > 0:
    labels.append(f'Ostatní ({len(layer_counts) - top_for_pie} vrstev)')
    sizes.append(other)

ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.set_title(f'Rozložení top {TOP_SCALARS} rozdílů mezi vrstvami')
plt.show()

In [36]:
# Buňka 8: Kopírování vah z modelu 1 do modelu 2

def copy_layer_weights(
    source_cpt: dict,
    target_cpt: dict,
    layers_to_copy: List[str]
) -> dict:
    """
    Zkopíruje váhy specifikovaných vrstev ze source do target modelu.
    
    Args:
        source_cpt: Zdrojový checkpoint
        target_cpt: Cílový checkpoint (bude modifikován)
        layers_to_copy: Seznam názvů vrstev k zkopírování
    
    Returns:
        Nový checkpoint s zkopírovanými váhami
    """
    # Kopie target checkpointu
    new_cpt = {
        "config": target_cpt["config"],
        "sr": target_cpt["sr"],
        "f0": target_cpt.get("f0", 1),
        "version": target_cpt.get("version", "v1"),
        "weight": OrderedDict(target_cpt["weight"]),
        "info": f"Hybrid model: copied {len(layers_to_copy)} layers from source"
    }
    
    copied_keys = []
    
    for layer_name in layers_to_copy:
        # Najdi všechny váhy pro tuto vrstvu
        weight_keys = layer_to_weight_keys(layer_name, source_cpt["weight"])
        
        for key in weight_keys:
            if key in source_cpt["weight"] and key in new_cpt["weight"]:
                src_weight = source_cpt["weight"][key]
                tgt_weight = new_cpt["weight"][key]
                
                # Kontrola kompatibility tvarů
                if src_weight.shape == tgt_weight.shape:
                    new_cpt["weight"][key] = src_weight.clone()
                    copied_keys.append(key)
                else:
                    print(f"  Přeskakuji {key}: různé tvary {src_weight.shape} vs {tgt_weight.shape}")
    
    return new_cpt, copied_keys

# Kopírování vah
print(f"Kopíruji váhy z modelu 1 ({MODEL_PATH_1}) do modelu 2 ({MODEL_PATH_2})")
print(f"Vrstvy k zkopírování: {len(unique_layers)}")
print("-" * 60)

new_cpt, copied_keys = copy_layer_weights(cpt1, cpt2, unique_layers)

print(f"\nZkopírováno {len(copied_keys)} weight tensorů:")
for key in copied_keys:
    print(f"  - {key}")

Kopíruji váhy z modelu 1 (Ravdess_F_Angry_60e_1140s.pth) do modelu 2 (Ravdess_F_Neutral_111e_1110s.pth)
Vrstvy k zkopírování: 10
------------------------------------------------------------

Zkopírováno 71 weight tensorů:
  - enc_p.encoder.attn_layers.0.conv_k.weight
  - enc_p.encoder.attn_layers.0.conv_k.bias
  - enc_p.encoder.attn_layers.0.conv_q.weight
  - enc_p.encoder.attn_layers.0.conv_q.bias
  - enc_p.encoder.attn_layers.0.conv_v.weight
  - enc_p.encoder.attn_layers.0.conv_v.bias
  - dec.conv_pre.weight
  - dec.conv_pre.bias
  - dec.resblocks.1.convs1.0.bias
  - dec.resblocks.1.convs1.0.weight_g
  - dec.resblocks.1.convs1.0.weight_v
  - dec.resblocks.1.convs1.1.bias
  - dec.resblocks.1.convs1.1.weight_g
  - dec.resblocks.1.convs1.1.weight_v
  - dec.resblocks.1.convs1.2.bias
  - dec.resblocks.1.convs1.2.weight_g
  - dec.resblocks.1.convs1.2.weight_v
  - dec.resblocks.1.convs2.0.bias
  - dec.resblocks.1.convs2.0.weight_g
  - dec.resblocks.1.convs2.0.weight_v
  - dec.resblocks.1.co

In [37]:
# Buňka 9: Uložení hybridního modelu

OUTPUT_PATH = "hybrid_model.pth"

# Uložení
torch.save(new_cpt, OUTPUT_PATH)
print(f"Hybridní model uložen do: {OUTPUT_PATH}")
print(f"Velikost souboru: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")

Hybridní model uložen do: hybrid_model.pth
Velikost souboru: 54.91 MB


In [15]:
# Buňka 10: Validace - načtení a test hybridního modelu

print("Načítám hybridní model pro validaci...")
model_hybrid, cpt_hybrid = load_rvc_model(OUTPUT_PATH, DEVICE)

# Test inference
print("\nTestuji inferenci...")
with torch.no_grad():
    output_hybrid = model_hybrid.infer(**inputs)

print(f"Output shape: {output_hybrid[0].shape}")

# Porovnání výstupů
diff_vs_model1 = torch.abs(output_hybrid[0] - output1[0]).mean().item()
diff_vs_model2 = torch.abs(output_hybrid[0] - output2[0]).mean().item()

print(f"\nPrůměrný rozdíl výstupu:")
print(f"  Hybrid vs Model 1 (source): {diff_vs_model1:.6f}")
print(f"  Hybrid vs Model 2 (target): {diff_vs_model2:.6f}")

Načítám hybridní model pro validaci...
Načten model: hybrid_model.pth
  - Sample rate: 48000
  - Version: v2
  - Use F0: True

Testuji inferenci...
Output shape: torch.Size([1, 1, 147840])

Průměrný rozdíl výstupu:
  Hybrid vs Model 1 (source): 0.029528
  Hybrid vs Model 2 (target): 0.027570


In [ ]:
# Buňka 11: Shrnutí

print("=" * 70)
print("SHRNUTÍ")
print("=" * 70)
print(f"\nZdrojový model (source): {MODEL_PATH_1}")
print(f"Cílový model (target):   {MODEL_PATH_2}")
print(f"Výstupní model:          {OUTPUT_PATH}")
print(f"\nMetoda:")
print(f"  1. Analyzováno top {TOP_SCALARS} skalárních rozdílů v aktivacích")
print(f"  2. Vybráno {TOP_N_LAYERS} vrstev s nejvíce rozdíly")
print(f"  3. Zkopírováno {len(copied_keys)} weight tensorů")
print(f"\nZkopírované vrstvy:")
for layer, count in top_n_layers:
    pct = count / TOP_SCALARS * 100
    print(f"  - {layer}: {count} skalárů ({pct:.1f}%)")